In [1]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"


from pprint import pformat

formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=100, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=100, sort_dicts=False)))

<function IPython.lib.pretty._seq_pprinter_factory.<locals>.inner(obj, p, cycle)>

In [2]:
from datasets import load_dataset, concatenate_datasets

In [25]:
zsre_raw = load_dataset('gjyotin305/full_zsre')
zsre_raw

zsre_concat = concatenate_datasets([zsre_raw[split] for split in zsre_raw.keys()])
zsre_concat

zsre_concat[0]

DatasetDict({
    train: Dataset({
        features: ['subject', 'src', 'pred', 'rephrase', 'alt', 'answers', 'loc', 'loc_ans', 'cond'],
        num_rows: 163196
    })
    test: Dataset({
        features: ['subject', 'src', 'pred', 'rephrase', 'alt', 'answers', 'loc', 'loc_ans', 'cond'],
        num_rows: 19086
    })
})

Dataset({
    features: ['subject', 'src', 'pred', 'rephrase', 'alt', 'answers', 'loc', 'loc_ans', 'cond'],
    num_rows: 182282
})

{'subject': 'Colt King Cobra',
 'src': 'The manufacturer of Colt King Cobra was who?',
 'pred': "Colt's Manufacturing Company",
 'rephrase': 'Which company did Colt King Cobra produce?',
 'alt': "Colt's Manufacturing Corporation",
 'answers': ["Colt's Manufacturing Company"],
 'loc': 'nq question: ek veer ki ardaas veera meaning in english',
 'loc_ans': "A Brother's Prayer... Veera",
 'cond': "Colt's Manufacturing Company >> Colt's Manufacturing Corporation || The manufacturer of "
         'Colt King Cobra was who?'}

In [35]:
zsre_final = zsre_concat\
.filter(lambda s: s['pred'] in s['answers'] and len(s['answers']) == 1)\
.map(lambda s: {
    'subject': s['subject'],
    'prompt': s['src'],
    'old_ans': s['pred'],
    'new_ans': s['alt'],
    'rephrases': [s['rephrase']],
    'locality': [{
        'prompt': s['loc'].replace('nq question: ', ''),
        'ans': s['loc_ans']
    }]
}, remove_columns=zsre_concat.column_names)

zsre_final

zsre_final[0]

Map:   0%|          | 0/99531 [00:00<?, ? examples/s]

Dataset({
    features: ['subject', 'prompt', 'old_ans', 'new_ans', 'rephrases', 'locality'],
    num_rows: 99531
})

{'subject': 'Colt King Cobra',
 'prompt': 'The manufacturer of Colt King Cobra was who?',
 'old_ans': "Colt's Manufacturing Company",
 'new_ans': "Colt's Manufacturing Corporation",
 'rephrases': ['Which company did Colt King Cobra produce?'],
 'locality': [{'ans': "A Brother's Prayer... Veera",
               'prompt': 'ek veer ki ardaas veera meaning in english'}]}

In [36]:
zsre_final.save_to_disk('../data/zsre')

Saving the dataset (0/1 shards):   0%|          | 0/99531 [00:00<?, ? examples/s]

In [11]:
cfact_raw = load_dataset('azhx/counterfact')
cfact_raw

cfact_concat = concatenate_datasets([cfact_raw[split] for split in cfact_raw.keys()])
cfact_concat

cfact_concat[0]

DatasetDict({
    train: Dataset({
        features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
        num_rows: 19728
    })
    test: Dataset({
        features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
        num_rows: 2191
    })
})

Dataset({
    features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
    num_rows: 21919
})

{'case_id': 0,
 'pararel_idx': 2796,
 'requested_rewrite': {'prompt': 'The mother tongue of {} is',
                       'relation_id': 'P103',
                       'subject': 'Danielle Darrieux',
                       'target_new': {'id': 'Q1860', 'str': 'English'},
                       'target_true': {'id': 'Q150', 'str': 'French'}},
 'paraphrase_prompts': ['Shayna does this and Yossel goes still and dies. Danielle Darrieux, a '
                        'native',
                        'An album was recorded for Capitol Nashville but never released. Danielle '
                        'Darrieux spoke the language'],
 'neighborhood_prompts': ['The mother tongue of Léon Blum is',
                          'The native language of Montesquieu is',
                          'François Bayrou, a native',
                          'The native language of Raymond Barre is',
                          'Michel Rocard is a native speaker of',
                          'Jacques Chaban-Delmas

In [12]:
cfact_final = cfact_concat\
.map(lambda s: {
    'subject': s['requested_rewrite']['subject'],
    'prompt': s['requested_rewrite']['prompt'].replace('{}', s['requested_rewrite']['subject']),
    'old_ans': s['requested_rewrite']['target_true']['str'],
    'new_ans': s['requested_rewrite']['target_new']['str'],
    'rephrases': list(set(s['generation_prompts'])),
    'locality': [{
        'prompt': prompt,
        'ans': s['requested_rewrite']['target_true']['str']
    } for prompt in list(set(s['neighborhood_prompts']))] + [{
        'prompt': prompt,
        'ans': s['requested_rewrite']['target_new']['str']
    } for prompt in list(set(s['attribute_prompts']))],
}, remove_columns=cfact_concat.column_names)\
.filter(lambda s: len(s['rephrases'])==3)

cfact_final

cfact_final[0]

Filter:   0%|          | 0/21919 [00:00<?, ? examples/s]

Dataset({
    features: ['subject', 'prompt', 'old_ans', 'new_ans', 'rephrases', 'locality'],
    num_rows: 20775
})

{'subject': 'Danielle Darrieux',
 'prompt': 'The mother tongue of Danielle Darrieux is',
 'old_ans': 'French',
 'new_ans': 'English',
 'rephrases': ["Danielle Darrieux's mother tongue is",
               'Danielle Darrieux was born in',
               'Where Danielle Darrieux is from, people speak the language of'],
 'locality': [{'ans': 'French', 'prompt': 'François Bayrou, a native'},
              {'ans': 'French', 'prompt': 'Maurice Genevoix, speaker of'},
              {'ans': 'French', 'prompt': 'Jacques Chaban-Delmas is a native speaker of'},
              {'ans': 'French', 'prompt': 'Michel Rocard is a native speaker of'},
              {'ans': 'French', 'prompt': 'The native language of Montesquieu is'},
              {'ans': 'French', 'prompt': 'The mother tongue of François Bayrou is'},
              {'ans': 'French', 'prompt': 'The native language of Raymond Barre is'},
              {'ans': 'French', 'prompt': 'The native language of François Bayrou is'},
              {'a

In [18]:
cfact_final.save_to_disk('../data/cfact')

Saving the dataset (0/1 shards):   0%|          | 0/20775 [00:00<?, ? examples/s]

In [9]:
from datasets import load_from_disk


cfact = load_from_disk('../data/cfact')


cfact.filter(lambda s: len(s['rephrases']) == 3)

Filter:   0%|          | 0/21919 [00:00<?, ? examples/s]

Dataset({
    features: ['subject', 'prompt', 'old_ans', 'new_ans', 'rephrases', 'locality'],
    num_rows: 20775
})

In [31]:
mquake_raw = load_dataset('henryzhongsc/MQuAKE-Remastered')
mquake_raw

mquake_cf = mquake_raw['CF9k']
mquake_cf

mquake_cf[0]

DatasetDict({
    CF3k: Dataset({
        features: ['case_id', 'questions', 'answer', 'answer_alias', 'new_answer', 'new_answer_alias', 'requested_rewrite', 'single_hops', 'new_single_hops', 'orig_triples', 'orig_triples_labeled', 'new_triples', 'new_triples_labeled', 'edit_triples', 'split'],
        num_rows: 3000
    })
    CF9k: Dataset({
        features: ['case_id', 'questions', 'answer', 'answer_alias', 'new_answer', 'new_answer_alias', 'requested_rewrite', 'single_hops', 'new_single_hops', 'orig_triples', 'orig_triples_labeled', 'new_triples', 'new_triples_labeled', 'edit_triples', 'split'],
        num_rows: 9171
    })
    CF6334: Dataset({
        features: ['case_id', 'questions', 'answer', 'answer_alias', 'new_answer', 'new_answer_alias', 'requested_rewrite', 'single_hops', 'new_single_hops', 'orig_triples', 'orig_triples_labeled', 'new_triples', 'new_triples_labeled', 'edit_triples', 'split'],
        num_rows: 9171
    })
    T: Dataset({
        features: ['case_id', '

Dataset({
    features: ['case_id', 'questions', 'answer', 'answer_alias', 'new_answer', 'new_answer_alias', 'requested_rewrite', 'single_hops', 'new_single_hops', 'orig_triples', 'orig_triples_labeled', 'new_triples', 'new_triples_labeled', 'edit_triples', 'split'],
    num_rows: 9171
})

{'case_id': 1,
 'questions': ['Who is the employer of the spouse of Ann Druyan?',
               'Where does the spouse of Ann Druyan work?',
               'What company employs the person married to Ann Druyan?'],
 'answer': 'Cornell University',
 'answer_alias': ['CU', 'Cornell', 'CUI', 'cornell.edu'],
 'new_answer': 'British Broadcasting Corporation',
 'new_answer_alias': ['British Broadcasting Corporation'],
 'requested_rewrite': [{'prompt': '{} is employed by',
                        'question': 'Who is the employer of Carl Sagan?',
                        'relation_id': 'P108',
                        'subject': 'Carl Sagan',
                        'target_new_id': 'Q9531',
                        'target_new_str': 'British Broadcasting Corporation',
                        'target_true_id': 'Q49115',
                        'target_true_str': 'Cornell University'}],
 'single_hops': [{'answer': 'Carl Sagan',
                  'answer_alias': ['Sagan CE',
                      

In [32]:
mquake_final = mquake_cf\
.filter(lambda s: len(s['single_hops']) == 2 and len(s['requested_rewrite']) == 1)\
.map(lambda s: {
    'subject': s['requested_rewrite'][0]['subject'],
    'prompt': s['requested_rewrite'][0]['question'],
    'old_ans': s['requested_rewrite'][0]['target_true_str'],
    'new_ans': s['requested_rewrite'][0]['target_new_str'],
    'portability': [{
        'prompt': s['questions'][0],
        'rephrases': s['questions'][1:],
        'old_ans': s['answer'],
        'new_ans': s['new_answer'],
        'hop': {
            'old': [{
                'prompt': h['question'],
                'ans': h['answer']
            } for h in s['single_hops']],
            'new': [{
                'prompt': h['question'],
                'ans': h['answer']
            } for h in s['new_single_hops']]
        }
    }]
}, remove_columns=mquake_cf.column_names)

mquake_final

mquake_final[0]

Dataset({
    features: ['subject', 'prompt', 'old_ans', 'new_ans', 'portability'],
    num_rows: 2446
})

{'subject': 'Carl Sagan',
 'prompt': 'Who is the employer of Carl Sagan?',
 'old_ans': 'Cornell University',
 'new_ans': 'British Broadcasting Corporation',
 'portability': [{'hop': {'new': [{'ans': 'Carl Sagan', 'prompt': 'Who is Ann Druyan married to?'},
                                  {'ans': 'British Broadcasting Corporation',
                                   'prompt': 'Who is the employer of Carl Sagan?'}],
                          'old': [{'ans': 'Carl Sagan', 'prompt': 'Who is Ann Druyan married to?'},
                                  {'ans': 'Cornell University',
                                   'prompt': 'Who is the employer of Carl Sagan?'}]},
                  'new_ans': 'British Broadcasting Corporation',
                  'old_ans': 'Cornell University',
                  'prompt': 'Who is the employer of the spouse of Ann Druyan?',
                  'rephrases': ['Where does the spouse of Ann Druyan work?',
                                'What company employs the

In [33]:
mquake_final.save_to_disk('../data/mquake')

Saving the dataset (0/1 shards):   0%|          | 0/2446 [00:00<?, ? examples/s]

In [34]:
# wiki = load_dataset('lukasthede/WikiBigEdit')['train']